In [362]:
import pandas as pd

In [392]:
df_transaction = pd.read_csv('./data/transaction/transaction_hsbc_dc.csv', sep='|')
df_transaction.loc[:, 'transaction_date'] = pd.to_datetime(df_transaction.loc[:, 'transaction_date'])
df_transaction = df_transaction.sort_values(by=['account', 'transaction_date'])
df_transaction.reset_index(drop=True, inplace=True)

df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2955 entries, 0 to 2954
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   account           2955 non-null   object        
 1   transaction_date  2955 non-null   datetime64[ns]
 2   payee             2915 non-null   object        
 3   master_category   2915 non-null   object        
 4   subcategory       2915 non-null   object        
 5   memo              624 non-null    object        
 6   amount            2955 non-null   float64       
 7   balance           2955 non-null   float64       
 8   end_of_day        40 non-null     object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 207.9+ KB


In [393]:
df_balance = pd.read_csv('./data/balance.csv', sep=',')
df_balance['statement_date'] = pd.to_datetime(df_balance['statement_date'])
df_balance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   statement_date   70 non-null     datetime64[ns]
 1   opening_balance  70 non-null     float64       
 2   closing_balance  70 non-null     float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 1.8 KB


In [305]:
# df_transaction.loc[:, 'balance'] = df_transaction['amount'].cumsum()
# df_transaction['eod_balance'] = df_transaction.groupby('transaction_date')['balance'].transform('last')
# df_transaction.loc[df_transaction.groupby('transaction_date').tail(1).index, 'end_of_day'] = 'True'
# df_transaction['end_of_day'] = df_transaction['end_of_day'].fillna('False')
# df_transaction.loc[df_transaction['end_of_day'] == 'False', 'eod_balance'] = 0

In [394]:
dt_list1 = list(set(list(df_transaction['transaction_date'])))
dt_list2 = list(set(list(df_balance['statement_date'])))
dt_list3 = [dt for dt in dt_list2 if dt not in dt_list1]

for dt in dt_list3:
    row = {'account': 'HSBC DC',
           'transaction_date': dt,
           'payee': '',
           'master_category': '',
           'subcategory': '',
           'memo': '',
           'amount': 0
          }
    df_transaction = df_transaction.append(row, ignore_index=True)
    
df_transaction = df_transaction.sort_values(by=['account', 'transaction_date'])
df_transaction.reset_index(drop=True, inplace=True)
df_transaction['balance'] = df_transaction['balance'].fillna(method='ffill')


In [399]:
df_transaction.loc[df_transaction.groupby('transaction_date').tail(1).index, 'end_of_day'] = 'True'
df_transaction['end_of_day'] = df_transaction['end_of_day'].fillna('False')

In [396]:
df_merged = pd.merge(df_transaction, df_balance, left_on=['transaction_date'], right_on=["statement_date"])
df_merged = df_merged[df_merged['end_of_day'] == 'True']
cols = ['transaction_date', 'payee', 'amount', 'balance', 'closing_balance']
df_merged = df_merged[cols]
df_merged['diff'] = round(df_merged['balance'] - df_merged['closing_balance'], 2)

In [397]:
df_merged.loc[df_merged['diff'] != 0]

,transaction_date,payee,amount,balance,closing_balance,diff


In [373]:
df_transaction_cc = pd.read_csv('./data/transaction/transaction_combined.csv', sep='|')
df_transaction_cc = df_transaction_cc[df_transaction_cc['account'] == 'HSBC CC']
df_transaction_cc.loc[:, 'transaction_date'] = pd.to_datetime(df_transaction_cc.loc[:, 'transaction_date'])
df_transaction_cc = df_transaction_cc.sort_values(by=['account', 'transaction_date'])
df_transaction_cc.reset_index(drop=True, inplace=True)
df_transaction_cc.loc[:, 'balance'] = round(df_transaction_cc['amount'].cumsum(),2)
df_transaction_cc.reset_index(drop=True, inplace=True)

In [375]:
df_tran.tail()

,account,transaction_date,payee,master_category,subcategory,memo,amount,balance
420,HSBC CC,2019-09-02,Transfer : HSBC DC,Transfer,In,NaN,364.38,-59.75
421,HSBC CC,2019-10-04,Transfer : HSBC DC,Transfer,In,NaN,71.53,11.78
422,HSBC CC,2019-12-09,Misc Discretionary,Long term,Misc Purchases,NaN,-11.78,-0.00
423,HSBC CC,2021-07-07,Carpetbright,Home improvement,House setup,Carpet cleaning,-162.00,-162.00
424,HSBC CC,2021-08-03,Transfer : HSBC DC,Transfer,In,NaN,162.00,-0.00


In [401]:
df_transaction.loc[(df_transaction['payee'].str.contains('Transfer')) & (df_transaction['payee'] != 'Transfer : HSBC CC'), :]

,account,transaction_date,payee,master_category,subcategory,memo,amount,balance,end_of_day
161,HSBC DC,2017-08-06,Transfer : HSBC Savings,Transfer : Out,HSBC Savings,NaN,-100.00,43071.56,True
1386,HSBC DC,2020-03-30,Transfer : ISA,Long term,ISA,NaN,-5000.00,91143.17,False
1387,HSBC DC,2020-03-30,Transfer : ISA,Long term,ISA,NaN,-5000.00,86143.17,False
1388,HSBC DC,2020-03-30,Transfer : ISA,Long term,ISA,NaN,-5000.00,81143.17,False
1389,HSBC DC,2020-03-30,Transfer : ISA,Long term,ISA,NaN,-5000.00,76143.17,False
1516,HSBC DC,2020-09-07,Transfer : ISA,Long term,ISA,NaN,-10000.00,62431.63,False
1522,HSBC DC,2020-09-07,Transfer : ISA,Long term,ISA,NaN,-10000.00,51571.85,True
2161,HSBC DC,2021-10-13,Transfer : Trading,Long term,Trading,Degiro,-0.01,8256.16,False
2166,HSBC DC,2021-10-15,Transfer : Trading,Long term,Trading,Degiro,-500.00,7658.39,True
2183,HSBC DC,2021-10-21,Transfer : Trading,Long term,Trading,Degiro,-20.00,7879.28,True


In [400]:
df_transaction = df_transaction.sort_values(by=['account', 'transaction_date'])
df_transaction.reset_index(drop=True, inplace=True)
df_transaction.to_csv('./data/transaction/transaction_hsbc_dc.csv', sep='|', index=False)

In [380]:
# df_transaction_cc.to_csv('./data/transaction/transaction_hsbc_cc.csv', sep='|', index=False)

In [386]:
df_transaction_cash = df_transaction_other.loc[df_transaction_other['account'] == 'Cash', :]

In [387]:
df_transaction_cash

,account,transaction_date,payee,master_category,subcategory,memo,amount
3077,Cash,2022-04-19,Transfer : HSBC DC,Transfer,In,NaN,30.0
3078,Cash,2022-03-01,Transfer : HSBC DC,Transfer,In,NaN,30.0
3079,Cash,2022-07-05,Transfer : HSBC DC,Transfer,In,NaN,30.0


In [388]:
# df_transaction_cash.to_csv('./data/transaction/transaction_cash.csv', sep='|', index=False)